In [2]:
# train_trie.py

from collections import Counter
from collections import defaultdict
import math
import numpy as np
import codecs
import tqdm
import time
import copy
import matplotlib.pyplot as plt
import pandas as pd
%matplotlib inline

from utils import iter_smiles
import trie_funcs as tf
from ape_tokenizer import APETokenizer
from SmilesPE.tokenizer import SPE_Tokenizer        

In [3]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

In [13]:
# load dataset and states

pubchem = "data/pubchem_100k_canonical.parquet"

TTG_FILE = "exp9_trie/ttg.pkl"
ttg_state = tf.load_state(TTG_FILE)

APE_DIR = "experiment6/ape"
ape_state = APETokenizer.from_pretrained(APE_DIR)
ape_state.load_vocabulary("experiment6/ape/vocab.json")

df = pd.read_parquet(pubchem)
smiles_list = df['SMILES'].tolist()


## TTG

In [14]:
# tokenize dataset

tokenized_smiles = []
for s in smiles_list:
    toks = tf.compress_and_return(s, ttg_state)
    tokenized_smiles.append(" ".join(toks))  # join tokens with spaces


In [15]:
# build TF–IDF feature extraction

vectorizer = TfidfVectorizer(
    analyzer="word",
    token_pattern=r"[^ ]+",   
    lowercase=False,
    max_features=5000          # adjust as needed
)
X_tfidf = vectorizer.fit_transform(tokenized_smiles)

print(f"TF-IDF matrix shape: {X_tfidf.shape}")  # (n_molecules, vocab_size)


TF-IDF matrix shape: (100000, 5000)


In [16]:
# PCA dimension reduction

svd = TruncatedSVD(n_components=50, random_state=42)
X_pca = svd.fit_transform(X_tfidf)  

print(f"SVD-reduced shape: {X_pca.shape}")

SVD-reduced shape: (100000, 50)


In [17]:
# k-means clustering

k = 20
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
labels = kmeans.fit_predict(X_pca)

In [18]:
# silhouette score computation

sil_score = silhouette_score(X_pca, labels)
print(f"Silhouette Score (k={k}): {sil_score:.4f}")

Silhouette Score (k=20): 0.5872


## APE

In [ ]:
# tokenize dataset

ape_tokenized_smiles = []
for s in smiles_list:
    ids = ape_state.encode(s)
    toks = ape_state.convert_ids_to_tokens(ids)
    ape_tokenized_smiles.append(" ".join(toks))


In [ ]:
# build TF–IDF feature extraction

vectorizer = TfidfVectorizer(
    analyzer="word",
    token_pattern=r"[^ ]+",   
    lowercase=False,
    max_features=5000        
)
X_tfidf = vectorizer.fit_transform(ape_tokenized_smiles)

print(f"TF-IDF matrix shape: {X_tfidf.shape}")  # (n_molecules, vocab_size)


In [ ]:
# PCA dimension reduction using TruncatedSVD

svd = TruncatedSVD(n_components=50, random_state=42)
X_pca = svd.fit_transform(X_tfidf)

print(f"SVD-reduced shape: {X_pca.shape}")

In [ ]:
# k-means clustering

k = 20
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
labels = kmeans.fit_predict(X_pca)

In [ ]:
# silhouette score computation

sil_score = silhouette_score(X_pca, labels)
print(f"Silhouette Score (k={k}): {sil_score:.4f}")

## Trie

In [4]:
# load dataset and states

pubchem = "data/pubchem_100k_canonical.parquet"

TRIE_FILE = "exp9_trie/trie.pkl"
trie_state = tf.load_state(TRIE_FILE)

df = pd.read_parquet(pubchem)
smiles_list = df['SMILES'].tolist()

In [5]:
# tokenize dataset

trie_tokenized_smiles = []
for s in smiles_list:
    toks = tf.compress_and_return(s, trie_state)
    trie_tokenized_smiles.append(" ".join(toks))  # join tokens with spaces

In [6]:
# build TF–IDF feature extraction

vectorizer = TfidfVectorizer(
    analyzer="word",
    token_pattern=r"[^ ]+",   
    lowercase=False,
    max_features=5000        
)
X_tfidf = vectorizer.fit_transform(trie_tokenized_smiles)

print(f"TF-IDF matrix shape: {X_tfidf.shape}")  # (n_molecules, vocab_size)


TF-IDF matrix shape: (100000, 5000)


In [7]:
# PCA dimension reduction using TruncatedSVD

svd = TruncatedSVD(n_components=50, random_state=42)
X_pca = svd.fit_transform(X_tfidf)

print(f"SVD-reduced shape: {X_pca.shape}")

SVD-reduced shape: (100000, 50)


In [8]:
# k-means clustering

k = 20
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
labels = kmeans.fit_predict(X_pca)

In [9]:
# silhouette score computation

sil_score = silhouette_score(X_pca, labels)
print(f"Silhouette Score (k={k}): {sil_score:.4f}")

Silhouette Score (k=20): 0.4723
